In [4]:
import pandas as pd
import numpy as np
import optuna
import gc
import warnings

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

PATH = "/kaggle/input/notebooks/luminhanh/ba-model-prep-data/"

X_TRAIN_PATH = PATH + "X_train.parquet"
Y_TRAIN_PATH = PATH + "y_train.parquet"

X_VAL_PATH = PATH + "X_val.parquet"
Y_VAL_PATH = PATH + "y_val.parquet"

# =========================================================
# OPTUNA CONFIG
# =========================================================

N_TRIALS = 30

# chỉ tune target đầu tiên
TARGET_COL = "target_day_1"

# dùng subset để tune nhanh
SAMPLE_SIZE = 200_000

RANDOM_STATE = 42

# =========================================================
# NWRMSLE
# =========================================================

def nwrmsle(y_true, y_pred, weights=None):

    y_pred = np.clip(y_pred, 0, None)

    if weights is None:
        weights = np.ones_like(y_true, dtype=np.float32)

    # y đã log1p từ preprocessing
    log_diff = y_pred - y_true

    weighted_mse = np.sum(
        weights * (log_diff ** 2)
    ) / np.sum(weights)

    return np.sqrt(weighted_mse)

# =========================================================
# LOAD DATA
# =========================================================

print("📥 Loading parquet files...")

X_train = pd.read_parquet(X_TRAIN_PATH)
y_train = pd.read_parquet(Y_TRAIN_PATH)

X_val = pd.read_parquet(X_VAL_PATH)
y_val = pd.read_parquet(Y_VAL_PATH)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

# =========================================================
# REMOVE NON NUMERIC
# =========================================================

drop_cols = []

for c in X_train.columns:

    dtype_str = str(X_train[c].dtype)

    if dtype_str in [
        "object",
        "datetime64[ns]"
    ]:
        drop_cols.append(c)

print("\n🗑️ Dropped columns:")
print(drop_cols)

X_train = X_train.drop(
    columns=drop_cols,
    errors="ignore"
)

X_val = X_val.drop(
    columns=drop_cols,
    errors="ignore"
)

# =========================================================
# CLEAN
# =========================================================

def clean_df(df):

    return (
        df
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
        .astype(np.float32)
    )

X_train = clean_df(X_train)
X_val = clean_df(X_val)

# =========================================================
# TARGET
# =========================================================

y_tr = y_train[
    TARGET_COL
].values.astype(np.float32)

y_va = y_val[
    TARGET_COL
].values.astype(np.float32)

# =========================================================
# VALIDATION WEIGHTS
# =========================================================

if "perishable" in X_val.columns:

    val_weights = (
        X_val["perishable"]
        .values
        .astype(np.float32) * 0.25
        + 1.0
    )

else:

    val_weights = np.ones(
        len(X_val),
        dtype=np.float32
    )

# =========================================================
# SCALE
# =========================================================

print("\n⚙️ Scaling features...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# =========================================================
# FREE RAM
# =========================================================

del X_train
del X_val

gc.collect()

# =========================================================
# CREATE SUBSET FOR OPTUNA
# =========================================================

print("\n🎯 Creating subset for Optuna...")

np.random.seed(RANDOM_STATE)

subset_idx = np.random.choice(
    len(X_train_scaled),
    SAMPLE_SIZE,
    replace=False
)

X_sub = X_train_scaled[subset_idx]
y_sub = y_tr[subset_idx]

print("X_sub:", X_sub.shape)
print("y_sub:", y_sub.shape)

# =========================================================
# OPTUNA OBJECTIVE
# =========================================================

def objective(trial):

    print("\n" + "="*70)
    print(f"🚀 START TRIAL {trial.number}")
    print("="*70)

    # =====================================================
    # HYPERPARAMETERS
    # =====================================================

    loss = trial.suggest_categorical(
        "loss",
        [
            "squared_error",
            "huber"
        ]
    )

    penalty = trial.suggest_categorical(
        "penalty",
        [
            "l2",
            "elasticnet"
        ]
    )

    alpha = trial.suggest_float(
        "alpha",
        1e-6,
        1e-4,
        log=True
    )

    eta0 = trial.suggest_float(
        "eta0",
        1e-4,
        1e-2,
        log=True
    )

    learning_rate = trial.suggest_categorical(
        "learning_rate",
        [
            "optimal",
            "adaptive"
        ]
    )

    l1_ratio = trial.suggest_float(
        "l1_ratio",
        0.0,
        1.0
    )

    max_iter = trial.suggest_int(
        "max_iter",
        20,
        100
    )

    tol = trial.suggest_float(
        "tol",
        1e-5,
        1e-3,
        log=True
    )

    # =====================================================
    # PRINT PARAMS
    # =====================================================

    print("\n📌 Hyperparameters")

    print("loss =", loss)
    print("penalty =", penalty)
    print("alpha =", alpha)
    print("eta0 =", eta0)
    print("learning_rate =", learning_rate)
    print("l1_ratio =", l1_ratio)
    print("max_iter =", max_iter)
    print("tol =", tol)

    # =====================================================
    # MODEL
    # =====================================================

    model = SGDRegressor(
        loss=loss,
        penalty=penalty,

        alpha=alpha,
        eta0=eta0,

        learning_rate=learning_rate,

        l1_ratio=l1_ratio,

        max_iter=max_iter,
        tol=tol,

        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,

        random_state=RANDOM_STATE
    )

    # =====================================================
    # SHUFFLE
    # =====================================================

    idx = np.random.permutation(
        len(X_sub)
    )

    X_shuf = X_sub[idx]
    y_shuf = y_sub[idx]

    # =====================================================
    # TRAIN
    # =====================================================

    print("\n⚙️ Training model...")

    model.fit(
        X_shuf,
        y_shuf
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    print("📊 Validating...")

    pred_val = model.predict(
        X_val_scaled
    )

    pred_val = np.clip(
        pred_val,
        0,
        None
    )

    score = nwrmsle(
        y_true=y_va,
        y_pred=pred_val,
        weights=val_weights
    )

    print(f"\n✅ Trial {trial.number} NWRMSLE: {score:.6f}")

    # =====================================================
    # CLEANUP
    # =====================================================

    del X_shuf
    del y_shuf
    del pred_val

    gc.collect()

    return score

# =========================================================
# CREATE STUDY
# =========================================================

print("\n🏁 Creating Optuna study...")

study = optuna.create_study(
    direction="minimize",
    study_name="favorita_sgd_fast"
)

# =========================================================
# START OPTUNA
# =========================================================

print("\n🚀 Starting optimization...\n")

study.optimize(
    objective,
    n_trials=N_TRIALS
)

# =========================================================
# BEST RESULT
# =========================================================

print("\n" + "="*80)
print("🏆 BEST PARAMETERS")
print("="*80)

for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\n🏆 BEST NWRMSLE")
print(study.best_value)

# =========================================================
# SAVE BEST PARAMS
# =========================================================

best_params_df = pd.DataFrame(
    [study.best_params]
)

best_params_df["best_score"] = (
    study.best_value
)

best_params_df.to_csv(
    "best_sgd_params.csv",
    index=False
)

print("\n✅ Saved: best_sgd_params.csv")

# =========================================================
# CLEAN RAM
# =========================================================

del X_sub
del y_sub

del X_train_scaled
del X_val_scaled

del y_train
del y_val

gc.collect()

print("\n🏆 OPTUNA DONE")

📥 Loading parquet files...
X_train: (1005090, 633)
y_train: (1005090, 16)
X_val: (167515, 633)
y_val: (167515, 16)

🗑️ Dropped columns:
[]

⚙️ Scaling features...

🎯 Creating subset for Optuna...


[I 2026-05-17 15:36:44,188] A new study created in memory with name: favorita_sgd_fast


X_sub: (200000, 633)
y_sub: (200000,)

🏁 Creating Optuna study...

🚀 Starting optimization...


🚀 START TRIAL 0

📌 Hyperparameters
loss = squared_error
penalty = elasticnet
alpha = 7.597486148242113e-06
eta0 = 0.0001982921926393791
learning_rate = optimal
l1_ratio = 0.5054908344993462
max_iter = 94
tol = 0.0004531491274392155

⚙️ Training model...


[I 2026-05-17 15:37:21,748] Trial 0 finished with value: 5036226486019.813 and parameters: {'loss': 'squared_error', 'penalty': 'elasticnet', 'alpha': 7.597486148242113e-06, 'eta0': 0.0001982921926393791, 'learning_rate': 'optimal', 'l1_ratio': 0.5054908344993462, 'max_iter': 94, 'tol': 0.0004531491274392155}. Best is trial 0 with value: 5036226486019.813.


📊 Validating...

✅ Trial 0 NWRMSLE: 5036226486019.813477

🚀 START TRIAL 1

📌 Hyperparameters
loss = huber
penalty = elasticnet
alpha = 5.174433318895994e-06
eta0 = 0.00013735096255010207
learning_rate = optimal
l1_ratio = 0.3491067210443457
max_iter = 40
tol = 0.00012880829247316388

⚙️ Training model...


...

📊 Validating...

✅ Trial 27 NWRMSLE: 0.547037

🚀 START TRIAL 28

📌 Hyperparameters
loss = squared_error
penalty = elasticnet
alpha = 5.546606983320032e-05
eta0 = 0.0019415089604956168
learning_rate = adaptive
l1_ratio = 0.5620981317990841
max_iter = 68
tol = 0.0007480269678953055

⚙️ Training model...


[I 2026-05-17 15:58:34,593] Trial 28 finished with value: 242294499.66073862 and parameters: {'loss': 'squared_error', 'penalty': 'elasticnet', 'alpha': 5.546606983320032e-05, 'eta0': 0.0019415089604956168, 'learning_rate': 'adaptive', 'l1_ratio': 0.5620981317990841, 'max_iter': 68, 'tol': 0.0007480269678953055}. Best is trial 7 with value: 0.5468820563670753.


📊 Validating...

✅ Trial 28 NWRMSLE: 242294499.660739

🚀 START TRIAL 29

📌 Hyperparameters
loss = squared_error
penalty = elasticnet
alpha = 5.951051732800564e-06
eta0 = 0.0008550149223730738
learning_rate = optimal
l1_ratio = 0.503251450505963
max_iter = 91
tol = 0.00041490439023792564

⚙️ Training model...


[I 2026-05-17 15:58:49,931] Trial 29 finished with value: 22556916939512.203 and parameters: {'loss': 'squared_error', 'penalty': 'elasticnet', 'alpha': 5.951051732800564e-06, 'eta0': 0.0008550149223730738, 'learning_rate': 'optimal', 'l1_ratio': 0.503251450505963, 'max_iter': 91, 'tol': 0.00041490439023792564}. Best is trial 7 with value: 0.5468820563670753.


📊 Validating...

✅ Trial 29 NWRMSLE: 22556916939512.203125

🏆 BEST PARAMETERS
loss: squared_error
penalty: elasticnet
alpha: 8.347031678630762e-05
eta0: 0.0009534848206710864
learning_rate: adaptive
l1_ratio: 0.642822558945543
max_iter: 70
tol: 0.0006347886278040432

🏆 BEST NWRMSLE
0.5468820563670753

✅ Saved: best_sgd_params.csv

🏆 OPTUNA DONE


In [5]:
# =========================================================
# FAVORITA - SGD MULTI HORIZON
# TRAIN 16 MODELS
# =========================================================

import pandas as pd
import numpy as np
import gc

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# =========================================================
# CONFIG
# =========================================================

PATH = "/kaggle/input/notebooks/luminhanh/ba-model-prep-data/"

X_TRAIN_PATH = PATH + "X_train.parquet"
Y_TRAIN_PATH = PATH + "y_train.parquet"

X_VAL_PATH = PATH + "X_val.parquet"
Y_VAL_PATH = PATH + "y_val.parquet"

X_TEST_PATH = PATH + "X_test.parquet"

TEST_ID_PATH = PATH + "test_id_matrix.pkl"

N_DAYS = 16

# =========================================================
# NWRMSLE METRIC
# =========================================================

def nwrmsle(y_true, y_pred, weights=None):

    # safety
    y_pred = np.clip(y_pred, 0, None)

    # default weights
    if weights is None:
        weights = np.ones_like(y_true, dtype=np.float32)

    # log error
    log_diff = y_pred - y_true

    # weighted mse
    weighted_mse = np.sum(
        weights * (log_diff ** 2)
    ) / np.sum(weights)

    return np.sqrt(weighted_mse)

# =========================================================
# LOAD DATA
# =========================================================

print("📥 Loading parquet files...")

X_train = pd.read_parquet(X_TRAIN_PATH)
y_train = pd.read_parquet(Y_TRAIN_PATH)

X_val = pd.read_parquet(X_VAL_PATH)
y_val = pd.read_parquet(Y_VAL_PATH)

X_test = pd.read_parquet(X_TEST_PATH)

test_id_matrix = pd.read_pickle(TEST_ID_PATH)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)

# =========================================================
# REMOVE NON NUMERIC
# =========================================================

drop_cols = []

for c in X_train.columns:

    dtype_str = str(X_train[c].dtype)

    if dtype_str in ["object", "datetime64[ns]"]:
        drop_cols.append(c)

print("\nDropped columns:")
print(drop_cols)

X_train = X_train.drop(columns=drop_cols, errors="ignore")
X_val = X_val.drop(columns=drop_cols, errors="ignore")
X_test = X_test.drop(columns=drop_cols, errors="ignore")

# =========================================================
# CLEAN
# =========================================================

def clean_dataframe(df):

    df = (
        df
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
        .astype(np.float32)
    )

    return df

X_train = clean_dataframe(X_train)
X_val = clean_dataframe(X_val)
X_test = clean_dataframe(X_test)

# =========================================================
# SCALE
# =========================================================

print("\n⚙️ Scaling features...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# free RAM
del X_train, X_val, X_test
gc.collect()

# =========================================================
# VALIDATION WEIGHTS
# Kaggle Favorita:
# perishable == 1 -> weight = 1.25
# otherwise -> weight = 1.0
# =========================================================

if "perishable" in y_val.columns:

    val_weights = (
        y_val["perishable"].values.astype(np.float32) * 0.25
        + 1.0
    )

else:

    # fallback
    val_weights = np.ones(len(y_val), dtype=np.float32)

# =========================================================
# TRAIN 16 MODELS
# =========================================================

models = []
val_scores = []

print("\n🚀 Training 16 SGD models...\n")

for day in range(N_DAYS):

    target_col = y_train.columns[day]

    print(f"\n========== DAY {day+1} ==========")

    # =====================================================
    # TARGET
    # =====================================================

    y_tr = y_train[target_col].values.astype(np.float32)
    y_va = y_val[target_col].values.astype(np.float32)

    # =====================================================
    # MODEL
    # =====================================================

    model = SGDRegressor(
        loss="squared_error",
        penalty="elasticnet",

        alpha=8.347031678630762e-05,
        l1_ratio=0.642822558945543,

        learning_rate="adaptive",
        eta0=0.0009534848206710864,

        max_iter=70,
        tol=0.0006347886278040432,

        random_state=42
    )

    # =====================================================
    # SHUFFLE
    # =====================================================

    idx = np.random.permutation(len(X_train_scaled))

    X_shuf = X_train_scaled[idx]
    y_shuf = y_tr[idx]

    # =====================================================
    # TRAIN
    # =====================================================

    model.fit(
        X_shuf,
        y_shuf
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    pred_val = model.predict(X_val_scaled)

    # clip negative log predictions
    pred_val = np.clip(pred_val, 0, None)

    # =====================================================
    # NWRMSLE
    # =====================================================

    score = nwrmsle(
        y_true=y_va,
        y_pred=pred_val,
        weights=val_weights
    )

    print(f"NWRMSLE: {score:.5f}")

    val_scores.append(score)
    models.append(model)

    del y_tr, y_va
    del X_shuf, y_shuf
    del pred_val

    gc.collect()

print("\n🏆 Mean NWRMSLE:", np.mean(val_scores))

# =========================================================
# PREDICT TEST
# =========================================================

print("\n📦 Predicting test...")

all_preds = []

for day in range(N_DAYS):

    pred = models[day].predict(X_test_scaled)

    # remove negative log predictions
    pred = np.clip(pred, 0, None)

    # inverse log1p
    pred = np.expm1(pred)

    # no negative sales
    pred = np.clip(pred, 0, None)

    all_preds.append(pred)

test_preds = np.stack(all_preds, axis=1)

print("Prediction shape:", test_preds.shape)

# =========================================================
# BUILD SUBMISSION FROM ORIGINAL test.csv
# =========================================================

print("\n📝 Building final submission...")

# =====================================================
# LOAD ORIGINAL TEST.CSV
# =====================================================

df_test = pd.read_csv(
    PATH + "test.csv",
    parse_dates=["date"]
)

print("Original test shape:", df_test.shape)

# =====================================================
# CREATE STORE-ITEM PAIRS
# =====================================================

pair_df = pd.DataFrame({
    "store_nbr": test_id_matrix.index.get_level_values(0),
    "item_nbr": test_id_matrix.index.get_level_values(1)
})

print("Unique pairs:", pair_df.shape)

# =====================================================
# ADD 16-DAY PREDICTIONS
# =====================================================

for d in range(N_DAYS):

    pair_df[f"pred_day_{d+1}"] = test_preds[:, d]

# =====================================================
# MAP DATE -> DAY INDEX
# =====================================================

test_dates = sorted(df_test["date"].unique())

date_to_day = {
    d: i
    for i, d in enumerate(test_dates)
}

df_test["day_idx"] = df_test["date"].map(date_to_day)

# =====================================================
# MERGE PREDICTIONS
# =====================================================

df_test = df_test.merge(
    pair_df,
    on=["store_nbr", "item_nbr"],
    how="left"
)

# =====================================================
# PICK CORRECT DAY PREDICTION
# =====================================================

pred_cols = [
    f"pred_day_{i+1}"
    for i in range(N_DAYS)
]

pred_matrix = df_test[pred_cols].values

row_idx = np.arange(len(df_test))

df_test["unit_sales"] = pred_matrix[
    row_idx,
    df_test["day_idx"].values
]

# =====================================================
# HANDLE COLD START
# =====================================================

df_test["unit_sales"] = (
    df_test["unit_sales"]
    .fillna(0)
)

# =====================================================
# FINAL SUBMISSION
# =====================================================

submission = df_test[
    ["id", "unit_sales"]
].copy()

submission["id"] = submission["id"].astype(np.int64)

submission = submission.sort_values("id")

print("\nSubmission rows:", len(submission))

assert len(submission) == 3370464

submission.to_csv(
    "submission_sgd.csv",
    index=False
)

print("\n✅ submission_sgd.csv saved!")

# =========================================================
# CLEAN RAM
# =========================================================

del X_train_scaled
del X_val_scaled
del X_test_scaled

del y_train
del y_val

del df_test
del pair_df

gc.collect()

print("\n🏆 ALL DONE")

📥 Loading parquet files...
X_train: (1005090, 633)
y_train: (1005090, 16)
X_val: (167515, 633)
y_val: (167515, 16)
X_test: (167515, 633)

Dropped columns:
[]

⚙️ Scaling features...

🚀 Training 16 SGD models...


========== DAY 1 ==========
NWRMSLE: 0.54608

========== DAY 2 ==========
NWRMSLE: 0.56901

========== DAY 3 ==========
NWRMSLE: 0.58252

========== DAY 4 ==========
NWRMSLE: 0.59428

========== DAY 5 ==========
NWRMSLE: 0.59639

========== DAY 6 ==========
NWRMSLE: 0.59815

========== DAY 7 ==========
NWRMSLE: 0.64256

========== DAY 8 ==========
NWRMSLE: 0.62398

========== DAY 9 ==========
NWRMSLE: 0.61426

========== DAY 10 ==========
NWRMSLE: 0.60885

========== DAY 11 ==========
NWRMSLE: 0.61271

========== DAY 12 ==========
NWRMSLE: 0.62192

========== DAY 13 ==========
NWRMSLE: 0.61402

========== DAY 14 ==========
NWRMSLE: 0.60517

========== DAY 15 ==========
NWRMSLE: 0.59553

========== DAY 16 ==========
NWRMSLE: 0.60868

🏆 Mean NWRMSLE: 0.6021321122128898

📦 Predic